In [1]:
import numpy as np
import re

In [24]:
with open("names.txt", "r", encoding="utf-8") as f:
    data = f.read()
# data = open('names.txt', 'r',encoding='utf-8').read()
# cleaned_data = re.sub(r'[^a-zA-Z\s\n]', '', data) # To read only english characters
cleaned_data = re.sub(r'[^\u0900-\u097F\s\n]', '', data)
chars = list(set(cleaned_data))
data_size, vocab_size = len(cleaned_data), len(chars)
print(f"data has {data_size} characters, {vocab_size} unique. ")
char_to_ix = {char:ix for ix, char in enumerate(chars)}
ix_to_char = {ix:char for ix, char in enumerate(chars)}
print(char_to_ix)
print(ix_to_char)

data has 1275 characters, 45 unique. 
{'अ': 0, 'ओ': 1, 'ा': 2, 'म': 3, 'े': 4, 'द': 5, 'ग': 6, 'र': 7, 'ो': 8, 'क': 9, 'व': 10, 'ौ': 11, 'ऐ': 12, 'न': 13, 'थ': 14, 'ह': 15, 'प': 16, 'श': 17, 'ट': 18, 'स': 19, 'ण': 20, 'य': 21, 'ू': 22, 'ि': 23, 'ु': 24, 'ै': 25, 'ी': 26, 'ं': 27, 'ध': 28, 'ृ': 29, 'ष': 30, 'ल': 31, 'ऋ': 32, 'इ': 33, 'ख': 34, 'घ': 35, 'ज': 36, 'ई': 37, 'च': 38, 'आ': 39, 'त': 40, 'उ': 41, 'भ': 42, '\n': 43, '्': 44}
{0: 'अ', 1: 'ओ', 2: 'ा', 3: 'म', 4: 'े', 5: 'द', 6: 'ग', 7: 'र', 8: 'ो', 9: 'क', 10: 'व', 11: 'ौ', 12: 'ऐ', 13: 'न', 14: 'थ', 15: 'ह', 16: 'प', 17: 'श', 18: 'ट', 19: 'स', 20: 'ण', 21: 'य', 22: 'ू', 23: 'ि', 24: 'ु', 25: 'ै', 26: 'ी', 27: 'ं', 28: 'ध', 29: 'ृ', 30: 'ष', 31: 'ल', 32: 'ऋ', 33: 'इ', 34: 'ख', 35: 'घ', 36: 'ज', 37: 'ई', 38: 'च', 39: 'आ', 40: 'त', 41: 'उ', 42: 'भ', 43: '\n', 44: '्'}


In [25]:
print(chars)
print("vocab_size: ",len(chars))

['अ', 'ओ', 'ा', 'म', 'े', 'द', 'ग', 'र', 'ो', 'क', 'व', 'ौ', 'ऐ', 'न', 'थ', 'ह', 'प', 'श', 'ट', 'स', 'ण', 'य', 'ू', 'ि', 'ु', 'ै', 'ी', 'ं', 'ध', 'ृ', 'ष', 'ल', 'ऋ', 'इ', 'ख', 'घ', 'ज', 'ई', 'च', 'आ', 'त', 'उ', 'भ', '\n', '्']
vocab_size:  45


In [26]:
# Hyper parameters
hidden_size = 100 
seq_length = 25 # no of steps to unroll rnn for
learning_rate = 1e-1

In [27]:
# model parameters
Wxh = np.random.randn(hidden_size,vocab_size)*0.01 # input to hidden
Whh = np.random.randn(hidden_size,hidden_size)*0.01 # hidden to hidden
Why = np.random.randn(vocab_size,hidden_size)*0.01 # hidden to output
bh = np.zeros((hidden_size,1))*0.01 # hidden bias
by = np.zeros((vocab_size,1))*0.01 # output bias

In [28]:
def lossFun(inputs, targets, hprev):
    """
    inputs, targets are both list of integers.
    For example, inputs is a list of integers representing characters, where the integers are indices of the characters in vocabulary 
    hprev is Hx1 array of initial hidden state
    returns the loss, gradients on model parameters, and last hidden state
    """
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = np.copy(hprev)
    loss = 0
    
    # Forward Pass
    # For each time step 
    for t in range(len(inputs)):
        xs[t] = np.zeros((vocab_size,1))
        # Creating one hot vector for t-th input character
        xs[t][inputs[t]] = 1 
        hs[t] = np.tanh(np.dot(Wxh, xs[t]) + np.dot(Whh, hs[t-1]) + bh) # hidden state
        ys[t] = np.dot(Why, hs[t]) + by # unnormalized log probabilities for next chars
        ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t])) # probabilities for next chars
        loss += -np.log(ps[t][targets[t],0]) # softmax cross-entropy loss
    
    # backward pass: compute gradients going backwards
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dhnext = np.zeros_like(hs[0])
    for t in reversed(range(len(inputs))):
        dy = np.copy(ps[t])
        dy[targets[t]] -= 1 # backprop into y
        dWhy += np.dot(dy, hs[t].T)
        dby += dy
        dh = np.dot(Why.T, dy) + dhnext # backprop into h
        dhraw = (1 - hs[t] * hs[t]) * dh # backprop through tanh nonlinearity
        dbh += dhraw
        dWxh += np.dot(dhraw, xs[t].T)
        dWhh += np.dot(dhraw, hs[t-1].T)
        dhnext = np.dot(Whh.T, dhraw)
    for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
        np.clip(dparam, -5, 5, out=dparam) # clip to mitigate exploding gradients
    return loss, dWxh, dWhh, dWhy, dbh, dby, hs[len(inputs)-1]

def sample(h, seed_ix, n):
    """
    sample a sequence of integers from the model
    h is memory state, seed_ix is seed letter for first time step
    """
    x = np.zeros((vocab_size, 1))
    x[seed_ix] = 1
    ixes = []
    for t in range(n):
        h = np.tanh(np.dot(Wxh,x) + np.dot(Whh,h) + bh)
        y = np.dot(Why, h) + by
        p = np.exp(y) / np.sum(np.exp(y))
        ix = np.random.choice(range(vocab_size), p=p.ravel())
        x = np.zeros((vocab_size,1))
        ixes.append(ix)
    return ixes
    
        
        

In [29]:
n, p =0, 0
mWxh, mWhh, mWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
mbh, mby = np.zeros_like(bh), np.zeros_like(by) # memory vairables for Adagrad
smooth_loss = -np.log(1.0/vocab_size) * seq_length # loss at iteration 0

while True:
    # prepare inputs
    if p+seq_length + 1 > len(data) or n == 0:
        hprev = np.zeros((hidden_size,1)) # reset RNN memory if it is end of the data or if it is the first iteration
        p = 0 # go from the start of the data
        inputs = [char_to_ix[ch] for ch in cleaned_data[p:p+seq_length]]
        print("\ninputs: ",inputs)
        targets = [char_to_ix[ch] for ch in cleaned_data[p+1:p+seq_length+1]]
        print("\ntargets: ",targets)
    
    # sample from the model now and then
    if n%100 == 0:
        sample_ix = sample(hprev, inputs[0], 200)
        txt = ''.join(ix_to_char[ix] for ix in sample_ix)
        print("text: ",txt)
        
    # Forward sequence length characters through the network and fetch gradient
    loss, dWxh, dWhh, dWhy, dbh, dby, hprev = lossFun(inputs, targets, hprev)
    smooth_loss = smooth_loss * 0.999 + loss * 0.001
    if n%100 == 0:
        print(f"Iter: {n}, loss: {smooth_loss}") # printing progress
        
    # perform parameter update with Adagrad
    for param, dparam, mem in zip([Wxh, Whh, Why, bh, by],
                                  [dWxh, dWhh, dWhy, dbh, dby],
                                  [mWxh, mWhh, mWhy, mbh, mby]):
        mem += dparam * dparam
        param += -learning_rate * dparam/np.sqrt(mem + 1e-8) # adagrade upgrade
        
    p += seq_length
    n += 1 


inputs:  [39, 5, 23, 40, 44, 21, 43, 0, 7, 44, 20, 10, 43, 1, 36, 19, 43, 19, 11, 7, 42, 43, 19, 44, 10]

targets:  [5, 23, 40, 44, 21, 43, 0, 7, 44, 20, 10, 43, 1, 36, 19, 43, 19, 11, 7, 42, 43, 19, 44, 10, 7]
text:  नगयउंदीहयौऋेतअ्
ंगऐणजइदध्ौटअोृशकआऐधवचओोभख्ैऋदअिैधखदअटकथपचषहुपथखयशषेखमचरघैऋउसयक
पलयचईऐलपृपषंिभीओऋदअयणगआआृऋऋचैईेनजततशम्ललभेकधधइभौवरकहल
ूधइइउणूउंओ
थे्उगोथघरथैथरघदखघओपौइजउयवेपओिकननरतशधचौऋतचषघोवकऐणोसशणंलउआ
Iter: 0, loss: 95.1665681789841

inputs:  [39, 5, 23, 40, 44, 21, 43, 0, 7, 44, 20, 10, 43, 1, 36, 19, 43, 19, 11, 7, 42, 43, 19, 44, 10]

targets:  [5, 23, 40, 44, 21, 43, 0, 7, 44, 20, 10, 43, 1, 36, 19, 43, 19, 11, 7, 42, 43, 19, 44, 10, 7]
text:  सिरस्व्
यौचौय
ऐौासथणससखलआओपधणहऋजसषनपइधुईोकपैंमटीलधेआचससीचमरपसकृृृूंधीोपूऐ
पऐहईलइआऐटनेेनोखचणटपथखउौषपलूउचई
गपपानपृउटचौधकैौचाुे
हहहूनथशऐईऋऋृऋाइणसथषनाैेीधषषचधाओइुचौममैउधचापकनपनचैैधूजइचाचयासनपसटखसगूपापधऐजन
Iter: 100, loss: 93.21737350006991

inputs:  [39, 5, 23, 40, 44, 21, 43, 0, 7, 44, 20, 10, 43, 1, 36, 19, 43, 19, 11, 7, 42, 43,

KeyboardInterrupt: 